# 02 — Tiền xử lý UCF-Crime CLIP Features
Memory-Masked-EVT-MIL | Dataset chuẩn hóa + Thống kê lớp

Sử dụng features có sẵn của VadCLIP, không trích xuất lại ảnh. Giữ nguyên Test chính thức; tách 10% Train theo video, seed 42. Không augmentation hoặc nhân bản dữ liệu; cân bằng bằng sampler khi huấn luyện. Notebook không sửa file NPY gốc. Chạy lần lượt các cell; lỗi dữ liệu phải được xử lý trước khi xuất kết quả.

In [ ]:
# Nếu thiếu thư viện, bỏ dấu # ở dòng dưới và chạy:
# %pip install numpy pandas scikit-learn matplotlib
from pathlib import Path
import re, json
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Colab: mount Drive nếu cần, rồi chỉnh ROOT.
# from google.colab import drive
# drive.mount('/content/drive')
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/processed/clip_features').is_dir()), Path.cwd())
FEATURES = ROOT/'data/processed/clip_features'
ANNOTATIONS = ROOT/'data/annotations'
TRAIN_LIST = ANNOTATIONS/'ucf_CLIP_rgb.csv'
TEST_LIST = ANNOTATIONS/'ucf_CLIP_rgbtest.csv'
SEED = 42
VAL_RATIO = 0.10
DIM = 512
EXPECTED_TRAIN, EXPECTED_TEST = 1610, 290
print('ROOT:', ROOT)
print('Features:', FEATURES)
assert FEATURES.is_dir(), 'Chỉnh ROOT hoặc FEATURES cho đúng đường dẫn.'


## 1. Kiểm tra cấu trúc và danh sách gốc
Nếu CSV của tác giả nằm nơi khác, chỉnh TRAIN_LIST và TEST_LIST ở cell trên. Hỗ trợ CSV có hoặc không có header, đường dẫn Windows/Linux. Không suy đoán Train/Test từ tên thư mục.

In [ ]:
files = sorted(FEATURES.rglob('*.npy'))
assert files, 'Không tìm thấy NPY.'
print('Số file NPY:', len(files))
for p in files[:10]: print(p.relative_to(FEATURES))
for p in [TRAIN_LIST, TEST_LIST]:
    assert p.is_file(), f'Thiếu danh sách chính thức: {p}'
    print('\n', p.name)
    print('\n'.join(p.read_text(encoding='utf-8-sig').splitlines()[:5]))


In [ ]:
# Chỉ nhận dạng ID video UCF-Crime; hậu tố crop/segment không tạo video mới.
CATEGORIES = ['Normal','Abuse','Arrest','Arson','Assault','Burglary','Explosion','Fighting','RoadAccidents','Robbery','Shooting','Shoplifting','Stealing','Vandalism']
pattern = re.compile(r'(Normal(?:_?Videos?)?(?:_?For_?Anomaly)?|Road_?Accidents?|Abuse|Arrest|Arson|Assault|Burglary|Explosion|Fighting|Robbery|Shooting|Shoplifting|Stealing|Vandalism)[_-]?(\d+)', re.I)
def video_info(path):
    name = str(path).replace(chr(92), '/').rsplit('/',1)[-1]
    m = pattern.search(name)
    if not m: raise ValueError(f'Không nhận dạng được video ID: {name}')
    prefix, number = m.groups()
    category = 'Normal' if prefix.lower().startswith('normal') else ('RoadAccidents' if prefix.lower().startswith('road') else next(c for c in CATEGORIES if c.lower()==prefix.lower()))
    return category + str(int(number)).zfill(3), category

by_name = defaultdict(list)
for p in files: by_name[p.name.casefold()].append(p)

def load_list(path, split):
    table = pd.read_csv(path, header=None, dtype=str, keep_default_na=False)
    rows=[]; errors=[]
    for i,row in table.iterrows():
        candidates=[v.strip() for v in row if str(v).strip().lower().endswith('.npy')]
        if not candidates:
            if i==0 and any(str(v).strip().lower() in ('path','feature_path','video_path','filename','file','label','index') for v in row): continue
            errors.append(f'Dòng {i+1}: không có đường dẫn NPY'); continue
        ref=candidates[0].replace(chr(92),'/'); name=ref.rsplit('/',1)[-1]
        matches=by_name.get(name.casefold(),[])
        if len(matches)>1:
            matches=[p for p in matches if ref.casefold().endswith('/'+p.relative_to(FEATURES).as_posix().casefold())]
        if len(matches)!=1:
            errors.append(f'Dòng {i+1}: {name}, số file khớp={len(matches)}'); continue
        try: vid, cat=video_info(name)
        except ValueError as e: errors.append(str(e)); continue
        rows.append(dict(video_id=vid,feature_path=matches[0].relative_to(FEATURES).as_posix(),category=cat,label=int(cat!='Normal'),split=split))
    if errors: raise ValueError(f'{path.name}: {len(errors)} lỗi.\n'+'\n'.join(errors[:20]))
    assert rows, f'Danh sách rỗng: {path}'
    return pd.DataFrame(rows).drop_duplicates(subset=['feature_path'])
train_all=load_list(TRAIN_LIST,'train')
test=load_list(TEST_LIST,'test')
assert not set(train_all.video_id)&set(test.video_id), 'Video trùng giữa Train và Test.'
assert not set(train_all.feature_path)&set(test.feature_path), 'Feature trùng giữa Train và Test.'
print('Train:', train_all.video_id.nunique(), 'Test:', test.video_id.nunique())
assert train_all.video_id.nunique()==EXPECTED_TRAIN and test.video_id.nunique()==EXPECTED_TEST, 'Số video khác chuẩn UCF-Crime. Kiểm tra ID, CSV, dữ liệu trước khi tiếp tục; chỉ chỉnh EXPECTED nếu có lý do rõ ràng.'


## 2. Kiểm tra ma trận đặc trưng
Kiểm tra mỗi file được tham chiếu: dạng (T, 512), dtype số thực, không NaN/Inf, T > 0. Không tự chuẩn hóa L2 vì có thể ảnh hưởng cơ chế sai số tái tạo.

In [ ]:
records=[]; errors=[]
for rel in sorted(set(train_all.feature_path)|set(test.feature_path)):
    try:
        a=np.load(FEATURES/rel, mmap_mode='r', allow_pickle=False)
        if a.ndim!=2 or a.shape[0]==0 or a.shape[1]!=DIM: raise ValueError(f'shape={a.shape}')
        if not np.issubdtype(a.dtype,np.floating): raise ValueError(f'dtype={a.dtype}')
        if not np.isfinite(a).all(): raise ValueError('NaN/Inf')
        records.append(dict(feature_path=rel,timesteps=int(a.shape[0]),dtype=str(a.dtype)))
    except Exception as e: errors.append(f'{rel}: {e}')
print('Hợp lệ:',len(records),'Lỗi:',len(errors))
assert not errors, '\n'.join(errors[:20])
meta=pd.DataFrame(records)
print(meta.timesteps.describe())
unused=set(p.relative_to(FEATURES).as_posix() for p in files)-set(meta.feature_path)
print('File NPY không được tham chiếu:',len(unused))


In [ ]:
# 3. Chia validation theo ID video, phân tầng theo category.
unique=train_all[['video_id','category','label']].drop_duplicates()
assert unique.video_id.is_unique
tr_ids, va_ids=train_test_split(unique.video_id, test_size=VAL_RATIO, random_state=SEED, stratify=unique.category)
train=train_all[train_all.video_id.isin(tr_ids)].copy()
val=train_all[train_all.video_id.isin(va_ids)].copy(); val['split']='val'
assert not set(train.video_id)&set(val.video_id)
assert not set(train.video_id)&set(test.video_id)
assert not set(val.video_id)&set(test.video_id)
assert len(train)+len(val)==len(train_all)
all_rows=pd.concat([train,val,test],ignore_index=True).merge(meta,on='feature_path',validate='one_to_one')
video_rows=all_rows[['video_id','category','label','split']].drop_duplicates()
assert video_rows.video_id.is_unique
print(pd.crosstab(video_rows.category,video_rows.split))


In [ ]:
# 4. Xuất CSV, JSON và biểu đồ thực tế
SPLITS=ROOT/'data/splits'; STATS=ROOT/'data/statistics'
SPLITS.mkdir(parents=True,exist_ok=True); STATS.mkdir(parents=True,exist_ok=True)
for split in ['train','val','test']:
    all_rows[all_rows.split==split].sort_values(['video_id','feature_path']).to_csv(SPLITS/f'{split}.csv',index=False)
counts=pd.crosstab(video_rows.category,video_rows.split).reindex(index=CATEGORIES,columns=['train','val','test'],fill_value=0)
counts['total']=counts.sum(axis=1)
counts.to_csv(STATS/'class_distribution.csv')
split_counts=video_rows.split.value_counts().reindex(['train','val','test'])
summary=dict(dataset='UCF-Crime',feature_source='VadCLIP',feature_dim=DIM,total_videos=int(len(video_rows)),num_classes=int(video_rows.category.nunique()),feature_files=int(len(all_rows)),unused_npy_files=len(unused),seed=SEED,val_ratio=VAL_RATIO,splits={k:int(v) for k,v in split_counts.items()},binary_distribution={str(k):int(v) for k,v in video_rows.label.value_counts().items()},augmentation='none',balancing='balanced video sampling planned at training time',train_list=TRAIN_LIST.name,test_list=TEST_LIST.name)
(STATS/'dataset_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
ax=counts[['train','val','test']].plot.bar(stacked=True,figsize=(12,5)); ax.set_ylabel('Videos'); ax.set_xlabel('Category'); ax.set_title('UCF-Crime class distribution'); plt.tight_layout(); plt.savefig(STATS/'class_distribution.png',dpi=160); plt.show()
ax=split_counts.plot.bar(figsize=(6,4),rot=0); ax.set_ylabel('Videos'); ax.set_title('Train / Validation / Test'); ax.bar_label(ax.containers[0]); plt.tight_layout(); plt.savefig(STATS/'split_distribution.png',dpi=160); plt.show()
print(json.dumps(summary,indent=2)); print('Output:',SPLITS,STATS)


## 5. Ghi chú bàn giao
- Các CSV dùng đường dẫn tương đối từ FEATURES; nhiều feature cùng video phải được nhóm khi huấn luyện MIL.
- Giữ nguyên nhãn thời gian / ground truth chính thức để đánh giá về sau. Notebook chưa kiểm tra căn chỉnh nhãn từng frame với sampling của tác giả.
- Validation chỉ có nhãn cấp video nếu không có nhãn thời gian bổ sung; dùng để chọn mô hình theo metric cấp video, không dùng Test để hiệu chỉnh.
- Không commit video và features lớn lên GitHub. Commit notebook, CSV, JSON và PNG sau khi chạy thành công.